## Light Scraping of SteamCharts

Installing all required libraries for scrapping & filtering

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path

from pathlib import Path
import time
import re
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / "data"
DATA_RAW = DATA / "raw"

DATA_RAW.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0","Accept-Language": "en-US,en;q=0.9",}


def get_top_steam_games(n=10):
    url = "https://steamcharts.com/top"

    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    games = []

    for link in soup.select("a[href^='/app/']"):
        href = link.get("href")
        name = link.get_text(strip=True)

        match = re.search(r"/app/(\d+)", href)

        if match:
            app_id = int(match.group(1))

            games.append({
                "app_id": app_id,
                "name": name,
                "url": f"https://steamcharts.com/app/{app_id}"
            })

        if len(games) >= n:
            break

    return pd.DataFrame(games)


def scrape_steamcharts_app(app_id, game_name=None):
    url = f"https://steamcharts.com/app/{app_id}"

    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()

    tables = pd.read_html(StringIO(response.text))

    df = tables[0].copy()

    df.columns = ["month", "avg_players", "gain", "pct_gain", "peak_players"]

    df["app_id"] = app_id
    df["game_name"] = game_name

    df["month"] = pd.to_datetime(df["month"], errors="coerce")

    for col in ["avg_players", "gain", "pct_gain", "peak_players"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
            .replace("nan", pd.NA)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def scrape_top_games_history(n=20, delay=1.5):
    top_games = get_top_steam_games(n)
    all_histories = []
    failed = []

    for _, row in top_games.iterrows():
        app_id = row["app_id"]
        game_name = row["name"]

        print(f"Scraping {game_name} ({app_id})...")

        try:
            game_df = scrape_steamcharts_app(app_id, game_name)
            all_histories.append(game_df)

        except Exception as e:
            print(f"Failed: {game_name} ({app_id}) — {e}")
            failed.append({
                "app_id": app_id,
                "game_name": game_name,
                "error": str(e)
            })

        time.sleep(delay)

    history_df = pd.concat(all_histories, ignore_index=True)
    failed_df = pd.DataFrame(failed)

    return top_games, history_df, failed_df


top_games, steam_history, failed = scrape_top_games_history(n=20)

top_games.to_csv(DATA_RAW / "top_steam_games.csv", index=False)
steam_history.to_csv(DATA_RAW / "steamcharts_top_games_history.csv", index=False)
failed.to_csv(DATA_RAW / "steamcharts_failed.csv", index=False)

print(top_games.head())
print(steam_history.head())
print(failed)

Scraping Counter-Strike 2 (730)...


/tmp/ipykernel_141182/985649172.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["month"] = pd.to_datetime(df["month"], errors="coerce")


Scraping Dota 2 (570)...


/tmp/ipykernel_141182/985649172.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["month"] = pd.to_datetime(df["month"], errors="coerce")


Scraping PUBG: BATTLEGROUNDS (578080)...


...

Scraping Overwatch® (2357570)...


/tmp/ipykernel_141182/985649172.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["month"] = pd.to_datetime(df["month"], errors="coerce")


Scraping Stardew Valley (413150)...


/tmp/ipykernel_141182/985649172.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["month"] = pd.to_datetime(df["month"], errors="coerce")


    app_id                 name                                  url
0      730     Counter-Strike 2      https://steamcharts.com/app/730
1      570               Dota 2      https://steamcharts.com/app/570
2   578080  PUBG: BATTLEGROUNDS   https://steamcharts.com/app/578080
3  3419430            Bongo Cat  https://steamcharts.com/app/3419430
4  2868840     Slay the Spire 2  https://steamcharts.com/app/2868840
       month  avg_players      gain  pct_gain  peak_players  app_id  \
0        NaT    954278.11 -20507.60     -2.10       1554184     730   
1 2026-04-01    974785.71 -94387.41     -8.83       1564830     730   
2 2026-03-01   1069173.12 -14921.47     -1.38       1717624     730   
3 2026-02-01   1084094.59  13650.81      1.28       1627561     730   
4 2026-01-01   1070443.78  79266.89      8.00       1654355     730   

          game_name  
0  Counter-Strike 2  
1  Counter-Strike 2  
2  Counter-Strike 2  
3  Counter-Strike 2  
4  Counter-Strike 2  
Empty DataFrame
Columns: []

In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0","Accept-Language": "en-US,en;q=0.9",}

def get_top_steam_games(n=20):
    url = "https://steamcharts.com/top"

    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    games = []

    for link in soup.select("a[href^='/app/']"):
        href = link.get("href")
        name = link.get_text(strip=True)

        match = re.search(r"/app/(\d+)", href)

        if match:
            games.append({
                "app_id": int(match.group(1)),
                "game_name": name,
                "url": f"https://steamcharts.com{href}"
            })

        if len(games) >= n:
            break

    return pd.DataFrame(games)


top_games = get_top_steam_games(n=20)

print(top_games)

#This lightscrap allows us to view the top 20 most played Steam Games.

     app_id                  game_name                                  url
0       730           Counter-Strike 2      https://steamcharts.com/app/730
1       570                     Dota 2      https://steamcharts.com/app/570
2    578080        PUBG: BATTLEGROUNDS   https://steamcharts.com/app/578080
3   3419430                  Bongo Cat  https://steamcharts.com/app/3419430
4   1172470              Apex Legends™  https://steamcharts.com/app/1172470
5   2868840           Slay the Spire 2  https://steamcharts.com/app/2868840
6    252490                       Rust   https://steamcharts.com/app/252490
7   2507950                Delta Force  https://steamcharts.com/app/2507950
8   2676230                      FiveM  https://steamcharts.com/app/2676230
9    431960           Wallpaper Engine   https://steamcharts.com/app/431960
10  2767030              Marvel Rivals  https://steamcharts.com/app/2767030
11   271590  Grand Theft Auto V Legacy   https://steamcharts.com/app/271590
12   552990 